# Angular 13+ Complete Upgrade Notes
### Instructor Reference — With Examples & Use Cases

---

## Table of Contents
1. [Angular 13 — Ivy Only, No More View Engine](#angular-13)
2. [Angular 14 — Standalone Components & Typed Forms](#angular-14)
3. [Angular 15 — Stable Standalone, Functional Guards, Image Directive](#angular-15)
4. [Angular 16 — Signals, Required Inputs, Router Inputs](#angular-16)
5. [Angular 17 — New Control Flow, Deferrable Views, Vite + esbuild](#angular-17)
6. [Angular 18+ — Zoneless, Material 3, Stable Signals](#angular-18)
7. [Migration Guide & Best Practices](#migration)

---

> **How to use these notes:** Each section contains the **What changed**, **Why it matters**, **Code examples**, and **Real-world use cases**.

# Angular 13 — Ivy Only, No More View Engine

## Overview
Angular 13 (released November 2021) marked a major milestone: **View Engine was completely removed**. Ivy became the **only** compilation and rendering engine.

---

## Key Changes

### 1. Ivy is Now Mandatory
- View Engine support dropped entirely
- All libraries must be Ivy-compatible (APF — Angular Package Format updated)
- Enables smaller bundle sizes, faster builds, and better debugging

### 2. IE11 Support Dropped
- Internet Explorer 11 is no longer supported
- Allows use of modern browser APIs (ES2017+ target)
- Removes polyfills, reducing bundle size

### 3. TypeScript 4.4 Support
- Improved type inference, `--useUnknownInCatchVariables` flag
- Better narrowing with `&&` and `||`

### 4. RxJS 7.4 Required
- More efficient, smaller bundle
- Better TypeScript support

### 5. Persistent Build Cache (Default Enabled)
- Angular CLI now caches builds on disk by default
- Rebuild times reduced by up to **40%**

### 6. Dynamic Component Creation — API Simplification
**Before Angular 13** (legacy, required ComponentFactoryResolver):
```typescript
// OLD way (Angular < 13)
const factory = this.resolver.resolveComponentFactory(MyComponent);
const componentRef = viewContainerRef.createComponent(factory);
```

**Angular 13+ (simplified):**
```typescript
// NEW way (Angular 13+)
const componentRef = viewContainerRef.createComponent(MyComponent);
// No factory resolver needed!
```

### 7. TestBed Teardown Behavior
- `TestBed` now tears down the testing module and DOM after each test by default
- Prevents memory leaks in unit tests

---

## Use Cases

| Feature | Use Case |
|---|---|
| Ivy-only | Faster builds in large enterprise apps |
| No IE11 | Use CSS Grid, Fetch API without polyfills |
| Persistent cache | CI/CD pipelines — faster rebuild on partial changes |
| Dynamic components | Modals, tooltips, dynamic widgets without factory boilerplate |

---

## Migration Steps: Angular 12 → 13
```bash
# Update Angular CLI and core
ng update @angular/core@13 @angular/cli@13

# Update Angular Material (if used)
ng update @angular/material@13
```

**Remove from `polyfills.ts`:**
```typescript
// DELETE these IE-specific polyfills:
import 'core-js/es/reflect';
import 'zone.js/dist/zone-patch-rxjs';
```

# Angular 14 — Standalone Components, Typed Reactive Forms & More

## Overview
Angular 14 (released June 2022) introduced **Standalone Components** (the biggest API change in years), **Typed Reactive Forms**, and improvements to the developer experience.

---

## Key Changes

### 1. Standalone Components (Developer Preview → Stable in v15)
Components, directives, and pipes can now exist **without NgModule**.

**Before (NgModule required):**
```typescript
// app.module.ts
@NgModule({
  declarations: [AppComponent, HeaderComponent],
  imports: [BrowserModule, CommonModule],
  bootstrap: [AppComponent]
})
export class AppModule {}
```

**After (Standalone):**
```typescript
// header.component.ts
@Component({
  selector: 'app-header',
  standalone: true,          // <-- No NgModule needed
  imports: [CommonModule],   // <-- Import dependencies directly
  template: `<h1>Hello {{ title }}</h1>`
})
export class HeaderComponent {
  title = 'Angular 14';
}
```

**Bootstrap standalone app (`main.ts`):**
```typescript
import { bootstrapApplication } from '@angular/platform-browser';
import { AppComponent } from './app/app.component';

bootstrapApplication(AppComponent, {
  providers: [
    importProvidersFrom(RouterModule.forRoot(routes)),
    provideHttpClient()
  ]
});
```

**Use Case:** Micro-frontends, shared UI libraries, lazy-loaded widgets that don't need full module overhead.

---

### 2. Typed Reactive Forms (Strictly Typed)
Previously, form controls returned `any`. Now they are **fully typed**.

**Before Angular 14 (untyped):**
```typescript
const form = new FormGroup({
  name: new FormControl(''),   // type: AbstractControl<any>
  age: new FormControl(0)
});

const name = form.get('name').value;  // type: any
```

**Angular 14+ (strictly typed):**
```typescript
const form = new FormGroup({
  name: new FormControl<string>(''),       // type: string | null
  age: new FormControl<number>(0)          // type: number | null
});

const name = form.controls.name.value;    // type: string | null ✅
const age = form.controls.age.value;      // type: number | null ✅
```

**Migration — use `UntypedFormControl` for gradual migration:**
```typescript
// Temporary escape hatch during migration
const legacy = new UntypedFormControl('');  // same as old FormControl
```

**Use Case:** Login forms, registration forms — get type safety and IDE autocomplete.

---

### 3. Page Title Strategy
Angular Router now has a built-in way to set `<title>` per route.

```typescript
const routes: Routes = [
  {
    path: 'home',
    component: HomeComponent,
    title: 'Home — My App'    // <-- Sets document.title automatically
  },
  {
    path: 'profile',
    component: ProfileComponent,
    title: 'User Profile'
  }
];
```

**Custom Title Strategy:**
```typescript
@Injectable({ providedIn: 'root' })
export class AppTitleStrategy extends TitleStrategy {
  override updateTitle(routerState: RouterStateSnapshot): void {
    const title = this.buildTitle(routerState);
    document.title = title ? `${title} | My Company` : 'My Company';
  }
}

// In providers:
{ provide: TitleStrategy, useClass: AppTitleStrategy }
```

**Use Case:** SEO, accessibility, and browser tab management in SPAs.

---

### 4. Extended Developer Diagnostics
New compiler checks warn about:
- `nullish coalescing on non-nullable fields`
- Optional chaining that can never be null

```typescript
// angular.json — enable extended checks
{
  "angularCompilerOptions": {
    "extendedDiagnostics": {
      "checks": {
        "nullishCoalescingNotNullable": "warning"
      }
    }
  }
}
```

---

### 5. Optional Injectors in Embedded Views
Pass injectors into `ViewContainerRef.createEmbeddedView()` and `createComponent()`.

```typescript
const customInjector = Injector.create({
  providers: [{ provide: MY_TOKEN, useValue: 'custom-value' }]
});

viewContainerRef.createComponent(MyComponent, { injector: customInjector });
```

---

## Migration Steps: Angular 13 → 14
```bash
ng update @angular/core@14 @angular/cli@14
ng update @angular/material@14

# Migrate forms to typed (optional, auto-migration available)
ng generate @angular/core:typed-forms
```

## Use Cases Summary

| Feature | Real-World Use Case |
|---|---|
| Standalone Components | Shared component libraries (npm packages) |
| Typed Forms | User registration, checkout forms with validation |
| Page Title Strategy | SPA with multiple routes needing SEO-friendly titles |
| Optional Injectors | Testing components with mock providers |

# Angular 15 — Stable Standalone, Functional Guards, Image Directive

## Overview
Angular 15 (released November 2022) stabilized Standalone APIs, introduced **Functional Router Guards**, the **Directive Composition API**, and the powerful **NgOptimizedImage** directive.

---

## Key Changes

### 1. Stable Standalone APIs
All standalone APIs (`standalone: true`, `bootstrapApplication`, `provideRouter`) are now **stable** (no longer developer preview).

**Full Standalone App Setup:**
```typescript
// main.ts
import { bootstrapApplication } from '@angular/platform-browser';
import { provideRouter } from '@angular/router';
import { provideHttpClient } from '@angular/common/http';
import { provideAnimations } from '@angular/platform-browser/animations';

bootstrapApplication(AppComponent, {
  providers: [
    provideRouter(routes),           // replaces RouterModule.forRoot()
    provideHttpClient(),             // replaces HttpClientModule
    provideAnimations(),             // replaces BrowserAnimationsModule
  ]
});
```

---

### 2. Functional Router Guards
Guards can now be **plain functions** instead of injectable classes.

**Before (class-based guard):**
```typescript
@Injectable({ providedIn: 'root' })
export class AuthGuard implements CanActivate {
  constructor(private auth: AuthService, private router: Router) {}

  canActivate(): boolean | UrlTree {
    return this.auth.isLoggedIn()
      ? true
      : this.router.createUrlTree(['/login']);
  }
}

// routes
{ path: 'dashboard', canActivate: [AuthGuard] }
```

**After (functional guard):**
```typescript
// auth.guard.ts
export const authGuard = () => {
  const auth = inject(AuthService);
  const router = inject(Router);

  return auth.isLoggedIn()
    ? true
    : router.createUrlTree(['/login']);
};

// routes
{ path: 'dashboard', canActivate: [authGuard] }
```

**Use Case:** Admin panels, authenticated dashboards — much less boilerplate.

---

### 3. Directive Composition API
Apply multiple directives to a component **via metadata** — no wrapping element needed.

```typescript
// tooltip.directive.ts
@Directive({ selector: '[appTooltip]', standalone: true })
export class TooltipDirective {
  @Input() appTooltip = '';
  // ...tooltip logic
}

// highlight.directive.ts
@Directive({ selector: '[appHighlight]', standalone: true })
export class HighlightDirective {
  @HostBinding('style.background') bg = 'yellow';
}

// button.component.ts — compose both directives!
@Component({
  selector: 'app-button',
  standalone: true,
  hostDirectives: [
    HighlightDirective,
    {
      directive: TooltipDirective,
      inputs: ['appTooltip: tooltip']   // alias the input
    }
  ],
  template: `<button><ng-content></ng-content></button>`
})
export class ButtonComponent {}
```

**Usage in template:**
```html
<app-button tooltip="Click to submit">Submit</app-button>
```

**Use Case:** Design system components that always need certain behaviors (tooltip, ripple, accessibility).

---

### 4. NgOptimizedImage Directive (Stable)
Automatic image optimization: lazy loading, priority hints, size warnings.

**Setup:**
```typescript
import { NgOptimizedImage } from '@angular/common';

@Component({
  standalone: true,
  imports: [NgOptimizedImage],
  template: `
    <!-- Basic usage -->
    <img ngSrc="hero.jpg" width="800" height="400" alt="Hero">

    <!-- Priority (LCP image — loads eagerly) -->
    <img ngSrc="banner.jpg" width="1200" height="600" priority alt="Banner">

    <!-- With image loader (e.g., Cloudinary) -->
    <img ngSrc="profile/user-123" fill alt="Profile">
  `
})
export class ImageComponent {}
```

**With CDN Loader:**
```typescript
// main.ts providers
import { provideImgixLoader } from '@angular/common';

bootstrapApplication(AppComponent, {
  providers: [
    provideImgixLoader('https://my-imgix-subdomain.imgix.net/')
  ]
});
```

**Use Case:** E-commerce product images, news article thumbnails — automatic Core Web Vitals improvement.

---

### 5. Stack Trace Improvements
Angular 15 provides **cleaner error stack traces** — framework internals are hidden, showing only your app code.

**Before:**
```
Error: Template parse error
  at ZoneDelegate.invoke (zone.js:372)
  at Object.onInvoke (core.mjs:26390)
  at ZoneDelegate.invoke (zone.js:371)
  at Zone.run (zone.js:134)
  ...15 framework frames...
  at AppComponent.doSomething (app.component.ts:42)
```

**After (Angular 15+):**
```
Error: Template parse error
  at AppComponent.doSomething (app.component.ts:42)
```

---

### 6. Router: `provideRouter` with Features
```typescript
import { provideRouter, withPreloading, PreloadAllModules, withDebugTracing } from '@angular/router';

bootstrapApplication(AppComponent, {
  providers: [
    provideRouter(
      routes,
      withPreloading(PreloadAllModules),     // preload lazy modules
      withDebugTracing(),                    // log router events (dev only)
      withRouterConfig({ paramsInheritanceStrategy: 'always' })
    )
  ]
});
```

---

## Migration Steps: Angular 14 → 15
```bash
ng update @angular/core@15 @angular/cli@15
ng update @angular/material@15
```

**Replace class-based guards (optional schematics):**
```bash
ng generate @angular/core:route-lazy-loading
```

## Use Cases Summary

| Feature | Real-World Use Case |
|---|---|
| Stable Standalone | Full modular-free applications |
| Functional Guards | Auth guard, role-based access, feature flags |
| Directive Composition | Design system (badges, tooltips on all interactive elements) |
| NgOptimizedImage | News sites, e-commerce — image performance |
| Clean Stack Traces | Faster debugging in production error logs |

# Angular 16 — Signals, Required Inputs, Router Inputs & SSR

## Overview
Angular 16 (released May 2023) introduced **Signals** (reactive primitive), **Required Inputs**, **Router-to-Component inputs binding**, and major SSR/hydration improvements.

---

## Key Changes

### 1. Signals (Developer Preview)
A new **reactive primitive** — a simpler alternative to RxJS for managing local state.

**Core Signal API:**
```typescript
import { signal, computed, effect } from '@angular/core';

@Component({
  standalone: true,
  template: `
    <p>Count: {{ count() }}</p>         <!-- call like a function -->
    <p>Double: {{ double() }}</p>
    <button (click)="increment()">+1</button>
  `
})
export class CounterComponent {
  // Create a signal
  count = signal(0);

  // Computed signal (derived, read-only)
  double = computed(() => this.count() * 2);

  increment() {
    this.count.update(v => v + 1);   // update based on previous value
    // OR: this.count.set(5);         // set to a specific value
  }

  constructor() {
    // Effect runs whenever signals it reads change
    effect(() => {
      console.log(`Count changed to: ${this.count()}`);
    });
  }
}
```

**Signal vs RxJS — When to use which:**
| Scenario | Signal | RxJS |
|---|---|---|
| Local UI state | ✅ Prefer | Possible |
| Async HTTP calls | Use with `toSignal()` | ✅ Prefer |
| Complex event streams | Less suited | ✅ Prefer |
| Component inputs | ✅ (v17+) | N/A |

**Interop with RxJS:**
```typescript
import { toSignal, toObservable } from '@angular/core/rxjs-interop';

@Component({ standalone: true })
export class SearchComponent {
  query = signal('');

  // Convert observable → signal
  results = toSignal(
    toObservable(this.query).pipe(
      debounceTime(300),
      switchMap(q => this.searchService.search(q))
    ),
    { initialValue: [] }
  );
}
```

---

### 2. Required Inputs
Input properties can now be marked as **required** at compile time.

**Before (runtime error only):**
```typescript
@Component({ selector: 'app-card' })
export class CardComponent {
  @Input() title: string = '';  // no compile-time enforcement
}
```

**Angular 16+ (compile-time error):**
```typescript
@Component({
  selector: 'app-card',
  standalone: true,
  template: `<h2>{{ title }}</h2>`
})
export class CardComponent {
  @Input({ required: true }) title!: string;  // compiler enforces this!
}
```

```html
<!-- ERROR at build time if title is missing -->
<app-card></app-card>                  <!-- ❌ Error: title is required -->
<app-card title="Hello"></app-card>    <!-- ✅ OK -->
```

**Use Case:** Design system components with mandatory props (Card, Modal, Button with label).

---

### 3. Router Inputs Binding
Bind **route parameters, query params, and data** directly to component `@Input()` — no need to inject `ActivatedRoute`.

**Enable in providers:**
```typescript
// main.ts
bootstrapApplication(AppComponent, {
  providers: [
    provideRouter(routes, withComponentInputBinding())  // <-- enable this
  ]
});
```

**Before (inject ActivatedRoute):**
```typescript
@Component({ standalone: true })
export class UserComponent implements OnInit {
  user!: User;

  constructor(private route: ActivatedRoute, private userService: UserService) {}

  ngOnInit() {
    const id = this.route.snapshot.paramMap.get('id')!;
    this.user = this.userService.getUser(id);
  }
}
```

**After (direct Input binding):**
```typescript
@Component({ standalone: true })
export class UserComponent {
  @Input() id!: string;           // bound from :id route param
  @Input() tab?: string;          // bound from ?tab= query param
  @Input('data') resolvedData!: User; // bound from route resolve data
}
```

**Route definition:**
```typescript
{
  path: 'users/:id',
  component: UserComponent,
  resolve: { data: userResolver }
}
```

---

### 4. Non-Destructive Hydration (SSR)
Angular 16 adds **client-side hydration** for SSR apps without re-rendering the DOM.

```typescript
// main.ts — enable hydration
import { provideClientHydration } from '@angular/platform-browser';

bootstrapApplication(AppComponent, {
  providers: [
    provideClientHydration()   // hydrate server-rendered DOM
  ]
});
```

**Benefits:**
- Eliminates the "flash of content" during hydration
- Faster Largest Contentful Paint (LCP)
- Works with Angular Universal

---

### 5. `takeUntilDestroyed` — Clean Subscription Management
```typescript
import { takeUntilDestroyed } from '@angular/core/rxjs-interop';

@Component({ standalone: true })
export class DataComponent {
  // No need for ngOnDestroy + Subject!
  constructor(private dataService: DataService) {
    dataService.getData()
      .pipe(takeUntilDestroyed())   // auto-unsubscribe on destroy
      .subscribe(data => console.log(data));
  }
}
```

---

### 6. `@Self`, `@SkipSelf`, `@Host` in `inject()`
```typescript
// Use DI flags directly in inject()
const service = inject(MyService, { self: true });
const parent = inject(MyService, { skipSelf: true });
const optional = inject(MyService, { optional: true });
```

---

## Migration Steps: Angular 15 → 16
```bash
ng update @angular/core@16 @angular/cli@16
ng update @angular/material@16
```

## Use Cases Summary

| Feature | Real-World Use Case |
|---|---|
| Signals | Shopping cart state, form field interactions |
| Required Inputs | Design system — enforce API contracts |
| Router Inputs | Product detail page with ID from URL |
| Hydration | Blog, news site with SSR + interactivity |
| takeUntilDestroyed | Data polling, WebSocket subscriptions |

# Angular 17 — New Control Flow, Deferrable Views, Vite + esbuild

## Overview
Angular 17 (released November 2023) was a **renaissance** release. It introduced a new **built-in control flow syntax** (`@if`, `@for`, `@switch`), **Deferrable Views** (`@defer`), a brand new **application builder** using Vite + esbuild, and a completely new [angular.dev](https://angular.dev) documentation site.

---

## Key Changes

### 1. New Built-In Control Flow Syntax
The old `*ngIf`, `*ngFor`, `*ngSwitch` directives are replaced with a **block-based syntax** built into the Angular compiler.

#### `@if` / `@else if` / `@else`
**Before (`*ngIf`):**
```html
<div *ngIf="isLoggedIn; else guestBlock">
  Welcome, {{ user.name }}!
</div>
<ng-template #guestBlock>
  <p>Please log in.</p>
</ng-template>
```

**After (`@if`):**
```html
@if (isLoggedIn) {
  <div>Welcome, {{ user.name }}!</div>
} @else if (isPending) {
  <p>Loading...</p>
} @else {
  <p>Please log in.</p>
}
```

---

#### `@for` with `@empty`
**Before (`*ngFor`):**
```html
<ul>
  <li *ngFor="let item of items; trackBy: trackById">{{ item.name }}</li>
</ul>
<p *ngIf="items.length === 0">No items found.</p>
```

**After (`@for` — `track` is REQUIRED):**
```html
<ul>
  @for (item of items; track item.id) {
    <li>{{ item.name }}</li>
  } @empty {
    <p>No items found.</p>
  }
</ul>
```

**Available loop variables:**
```html
@for (item of items; track item.id; let i = $index, last = $last, even = $even) {
  <div [class.last]="last" [class.even]="even">
    {{ i + 1 }}. {{ item.name }}
  </div>
}
```

| Variable | Meaning |
|---|---|
| `$index` | Current index (0-based) |
| `$count` | Total items count |
| `$first` | `true` if first item |
| `$last` | `true` if last item |
| `$even` | `true` if even index |
| `$odd` | `true` if odd index |

---

#### `@switch` / `@case` / `@default`
```html
@switch (userRole) {
  @case ('admin') {
    <app-admin-panel />
  }
  @case ('editor') {
    <app-editor-tools />
  }
  @default {
    <app-viewer />
  }
}
```

---

### 2. Deferrable Views (`@defer`)
Load components **lazily** — only when needed. Huge performance win for below-the-fold content.

**Basic usage:**
```html
@defer {
  <app-heavy-chart />          <!-- loaded lazily -->
} @placeholder {
  <p>Loading chart...</p>
} @loading (minimum 500ms) {
  <app-spinner />
} @error {
  <p>Failed to load chart.</p>
}
```

**Defer triggers:**
```html
<!-- On viewport — load when visible -->
@defer (on viewport) {
  <app-comments-section />
}

<!-- On interaction — load when user clicks/focuses -->
@defer (on interaction) {
  <app-like-button />
}

<!-- On idle — load when browser is idle -->
@defer (on idle) {
  <app-analytics-widget />
}

<!-- On timer — load after 2 seconds -->
@defer (on timer(2s)) {
  <app-newsletter-popup />
}

<!-- On hover -->
@defer (on hover) {
  <app-tooltip-content />
}

<!-- Prefetch separately from render -->
@defer (on viewport; prefetch on idle) {
  <app-product-recommendations />
}
```

**Use Case — E-commerce Product Page:**
```html
<app-product-hero />                  <!-- loads immediately -->

@defer (on viewport) {
  <app-product-reviews />             <!-- lazy: below fold -->
} @placeholder {
  <div class="review-skeleton"></div>
}

@defer (on idle; prefetch on viewport) {
  <app-related-products />            <!-- lazy: low priority -->
}
```

---

### 3. New Application Builder (Vite + esbuild)
Angular 17 uses **Vite for dev server** and **esbuild for production builds** by default.

**Benefits:**
| Metric | Webpack (old) | esbuild (new) |
|---|---|---|
| Cold build | ~45s | ~5s |
| Incremental rebuild | ~3s | ~200ms |
| HMR (Hot Module Reload) | Supported | Faster |

**`angular.json` — new builder:**
```json
{
  "architect": {
    "build": {
      "builder": "@angular-devkit/build-angular:application",
      "options": {
        "browser": "src/main.ts",
        "server": "src/main.server.ts",   // SSR entry point (optional)
        "ssr": {
          "entry": "server.ts"
        }
      }
    }
  }
}
```

---

### 4. Signal Inputs (v17.1+)
```typescript
import { input, output, model } from '@angular/core';

@Component({ standalone: true })
export class UserCardComponent {
  // Signal-based input (read-only signal)
  name = input<string>('');
  age = input.required<number>();       // required signal input

  // Signal-based output
  clicked = output<void>();

  // Two-way binding via model()
  value = model<string>('');           // replaces [(value)] pattern
}
```

**Template usage:**
```html
<app-user-card
  [name]="userName"
  [age]="userAge"
  (clicked)="onCardClick()"
  [(value)]="searchTerm"
/>
```

---

### 5. View Transitions API
```typescript
// app.config.ts
import { provideRouter, withViewTransitions } from '@angular/router';

bootstrapApplication(AppComponent, {
  providers: [
    provideRouter(routes, withViewTransitions())  // smooth page transitions
  ]
});
```

---

## Migration Steps: Angular 16 → 17
```bash
ng update @angular/core@17 @angular/cli@17
ng update @angular/material@17

# Migrate *ngIf/*ngFor to new control flow (automated!)
ng generate @angular/core:control-flow
```

## Use Cases Summary

| Feature | Real-World Use Case |
|---|---|
| `@if`/`@for`/`@switch` | Any template — cleaner, better type narrowing |
| `@defer` | News articles, dashboards, e-commerce pages |
| Vite + esbuild | Fast local development & CI builds |
| Signal Inputs | Reactive component APIs without zone.js |
| View Transitions | SPA page animations (list → detail) |

# Angular 18+ — Zoneless, Stable Signals, Material 3 & Beyond

## Overview
Angular 18 (May 2024), 19 (November 2024), and beyond push Angular toward a **zoneless future**, **stable Signals**, **incremental hydration**, and **Material 3** design system support.

---

## Angular 18 Key Changes

### 1. Zoneless Change Detection (Developer Preview)
Angular has relied on `zone.js` to detect changes since Angular 2. Zone.js patches browser APIs (setTimeout, Promises, etc.) which has performance costs. Angular 18 introduces **experimental zoneless** mode.

**Enable zoneless:**
```typescript
// main.ts
import { provideExperimentalZonelessChangeDetection } from '@angular/core';

bootstrapApplication(AppComponent, {
  providers: [
    provideExperimentalZonelessChangeDetection()
    // Also remove zone.js from angular.json polyfills!
  ]
});
```

**`angular.json` — remove zone.js:**
```json
{
  "polyfills": []   // remove "zone.js" from here
}
```

**How it works with Signals:**
```typescript
@Component({
  standalone: true,
  template: `<p>{{ count() }}</p><button (click)="inc()">+</button>`
})
export class CounterComponent {
  count = signal(0);   // Signal automatically schedules change detection
  inc() { this.count.update(v => v + 1); }
  // No zone.js needed — Angular knows exactly WHAT changed
}
```

**Benefits of Zoneless:**
- Smaller bundle (~13KB less without zone.js)
- Better performance (no monkey-patching)
- Works well with Web Workers
- More predictable change detection

---

### 2. Stable Signals API
Angular 18 stabilizes the core Signals API:
- `signal()`, `computed()`, `effect()` — **stable**
- Signal Inputs (`input()`, `input.required()`) — **stable**
- Signal Outputs (`output()`) — **stable**
- `model()` for two-way binding — **stable**

**Full Signal-based component example:**
```typescript
import { Component, signal, computed, input, output, model, effect } from '@angular/core';

@Component({
  selector: 'app-product',
  standalone: true,
  template: `
    <h2>{{ name() }}</h2>
    <p>Price: {{ formattedPrice() }}</p>
    <p>Qty: <button (click)="decQty()">-</button>
            {{ qty() }}
            <button (click)="incQty()">+</button></p>
    <p>Total: {{ total() }}</p>
    <button (click)="addToCart.emit({ name: name(), total: total() })">
      Add to Cart
    </button>
  `
})
export class ProductComponent {
  name = input.required<string>();
  price = input.required<number>();
  qty = model<number>(1);                   // two-way bindable
  addToCart = output<{ name: string; total: number }>();

  formattedPrice = computed(() => `$${this.price().toFixed(2)}`);
  total = computed(() => this.price() * this.qty());

  incQty() { this.qty.update(q => q + 1); }
  decQty() { this.qty.update(q => Math.max(1, q - 1)); }

  constructor() {
    effect(() => console.log('Total updated:', this.total()));
  }
}
```

---

### 3. Incremental Hydration (Angular 19)
Combines **SSR** with **`@defer`** — server renders static HTML, and JS hydrates sections **on demand**.

```html
<!-- This section is server-rendered but only hydrated when visible -->
@defer (hydrate on viewport) {
  <app-product-reviews [productId]="id" />
} @placeholder {
  <!-- Shows server HTML until hydration triggers -->
  <app-reviews-skeleton />
}
```

**Enable in providers:**
```typescript
import { provideClientHydration, withIncrementalHydration } from '@angular/platform-browser';

bootstrapApplication(AppComponent, {
  providers: [
    provideClientHydration(
      withIncrementalHydration()   // defer hydration until needed
    )
  ]
});
```

---

### 4. Angular Material 3 (M3)
Angular Material updated to follow Google's **Material Design 3** spec.

```typescript
// app.config.ts — use M3 theme
import { provideAnimationsAsync } from '@angular/platform-browser/animations/async';

bootstrapApplication(AppComponent, {
  providers: [provideAnimationsAsync()]
});
```

**M3 Theme setup (in `styles.scss`):**
```scss
@use '@angular/material' as mat;

// Define M3 color scheme
$theme: mat.define-theme((
  color: (
    theme-type: light,
    primary: mat.$violet-palette,
    tertiary: mat.$orange-palette,
  ),
  typography: (
    brand-family: 'Roboto',
    bold-weight: 900,
  ),
  density: (
    scale: -1,
  )
));

html {
  @include mat.all-component-themes($theme);
}
```

---

### 5. Route Redirects as Functions (Angular 19)
Dynamic redirect logic based on user state.

```typescript
const routes: Routes = [
  {
    path: '',
    redirectTo: () => {
      const auth = inject(AuthService);
      return auth.isLoggedIn() ? '/dashboard' : '/home';
    }
  }
];
```

---

### 6. `linkedSignal` — Dependent State
Create a signal that **resets** when a source signal changes.

```typescript
import { signal, linkedSignal } from '@angular/core';

@Component({ standalone: true })
export class ShippingComponent {
  country = signal('US');

  // Resets to 'standard' whenever country changes
  shippingMethod = linkedSignal(() =>
    this.country() === 'US' ? 'standard' : 'international'
  );
}
```

---

### 7. `resource()` API — Async Data Loading
```typescript
import { signal, resource } from '@angular/core';
import { HttpClient } from '@angular/common/http';

@Component({ standalone: true })
export class UserComponent {
  userId = signal(1);

  userResource = resource({
    request: () => this.userId(),
    loader: ({ request: id }) =>
      fetch(`/api/users/${id}`).then(r => r.json())
  });

  // Template usage:
  // userResource.value() — the data
  // userResource.isLoading() — loading state
  // userResource.error() — error state
}
```

---

## Angular Version Quick Reference

| Version | Release | Key Feature |
|---|---|---|
| Angular 13 | Nov 2021 | Ivy only, No IE11, Simpler dynamic components |
| Angular 14 | Jun 2022 | Standalone components, Typed forms |
| Angular 15 | Nov 2022 | Stable standalone, Functional guards, NgOptimizedImage |
| Angular 16 | May 2023 | Signals (preview), Required inputs, Router inputs |
| Angular 17 | Nov 2023 | `@if/@for/@defer`, Vite+esbuild, Signal inputs |
| Angular 18 | May 2024 | Zoneless (preview), Stable signals |
| Angular 19 | Nov 2024 | Incremental hydration, `linkedSignal`, `resource()` |
| Angular 20 | May 2025 | Stable zoneless, `resource()` stable |

# Migration Guide & Best Practices

## Step-by-Step Upgrade Strategy

### Rule #1 — Upgrade One Major Version at a Time
```
12 → 13 → 14 → 15 → 16 → 17 → 18 → 19
```

### Universal Upgrade Command
```bash
# Always use ng update — never manually edit package.json versions
ng update @angular/core @angular/cli

# For a specific version
ng update @angular/core@16 @angular/cli@16

# Preview what will change (dry run)
ng update --dry-run
```

---

## Migration Checklist: Legacy NgModule → Modern Standalone

### Step 1 — Migrate to Standalone Components
```bash
# Automated schematic (migrates declarations to standalone)
ng generate @angular/core:standalone
```

This schematic will:
- Add `standalone: true` to components/directives/pipes
- Move `imports` from `NgModule.imports` to each component
- Remove empty `NgModule` files

### Step 2 — Migrate Root Module
```bash
ng generate @angular/core:standalone --mode=convert-to-standalone
ng generate @angular/core:standalone --mode=prune-ng-modules
ng generate @angular/core:standalone --mode=standalone-bootstrap
```

### Step 3 — Migrate `*ngIf` / `*ngFor` to New Control Flow (Angular 17+)
```bash
ng generate @angular/core:control-flow
```

### Step 4 — Migrate Class Guards to Functional Guards
```bash
ng generate @angular/core:route-lazy-loading
```

### Step 5 — Migrate to Typed Reactive Forms
```bash
ng generate @angular/core:typed-forms
```

---

## Best Practices

### 1. Prefer `inject()` over Constructor Injection
```typescript
// OLD — constructor injection
@Component({})
export class MyComponent {
  constructor(
    private http: HttpClient,
    private auth: AuthService,
    private router: Router
  ) {}
}

// NEW — inject() function (works in constructors, factories, and guards)
@Component({})
export class MyComponent {
  private http = inject(HttpClient);
  private auth = inject(AuthService);
  private router = inject(Router);
}
```

### 2. Use `provideHttpClient` with features
```typescript
import { provideHttpClient, withInterceptors, withFetch } from '@angular/common/http';

bootstrapApplication(AppComponent, {
  providers: [
    provideHttpClient(
      withFetch(),                          // use Fetch API instead of XMLHttpRequest
      withInterceptors([authInterceptor])   // functional interceptors
    )
  ]
});

// Functional interceptor
export const authInterceptor: HttpInterceptorFn = (req, next) => {
  const token = inject(AuthService).getToken();
  return next(req.clone({ setHeaders: { Authorization: `Bearer ${token}` } }));
};
```

### 3. Lazy Loading with Standalone Routes
```typescript
// routes.ts
const routes: Routes = [
  {
    path: 'admin',
    loadChildren: () =>
      import('./admin/admin.routes').then(m => m.ADMIN_ROUTES)
  },
  {
    path: 'profile',
    loadComponent: () =>
      import('./profile/profile.component').then(m => m.ProfileComponent)
  }
];
```

### 4. Signal Best Practices
```typescript
// ✅ DO — use computed() for derived values
const fullName = computed(() => `${firstName()} ${lastName()}`);

// ❌ DON'T — use effect() to update other signals (creates cycles)
effect(() => { fullName.set(firstName() + lastName()); }); // BAD

// ✅ DO — use effect() for SIDE EFFECTS (logging, localStorage, DOM)
effect(() => {
  localStorage.setItem('theme', theme());
});

// ✅ DO — use untracked() to avoid tracking unnecessary reads
effect(() => {
  const val = count();  // tracked
  untracked(() => {
    console.log('other info:', otherSignal()); // NOT tracked
  });
});
```

### 5. Performance Checklist
```typescript
// ✅ Always use track in @for
@for (item of items; track item.id) { ... }

// ✅ Use @defer for below-fold content
@defer (on viewport) { <app-heavy-component /> }

// ✅ Use NgOptimizedImage for all <img> tags
<img ngSrc="hero.jpg" width="1200" height="600" priority alt="Hero">

// ✅ Use OnPush change detection strategy (even with zones)
@Component({
  changeDetection: ChangeDetectionStrategy.OnPush
})

// ✅ Use loadComponent for route-level lazy loading
{ path: 'detail', loadComponent: () => import('./detail.component') }
```

---

## Common Interview Questions & Answers

### Q: What is Ivy? Why does it matter?
> **Ivy** is Angular's current compilation and rendering engine (replaced View Engine in v9, mandatory from v13). It enables better tree-shaking, faster builds, smaller bundles, and improved debugging.

### Q: What are Signals and why were they introduced?
> **Signals** are a reactive primitive that allow Angular to track **what** changed (fine-grained reactivity) instead of **checking everything** (zone.js dirty-checking). They enable zoneless apps and better performance.

### Q: What's the difference between `@defer` triggers?
| Trigger | When JS loads |
|---|---|
| `on idle` | Browser idle (requestIdleCallback) |
| `on viewport` | Element enters viewport |
| `on interaction` | User clicks/focuses the placeholder |
| `on hover` | User hovers over placeholder |
| `on timer(Xs)` | After X seconds |
| `on immediate` | Right away (async, non-blocking) |

### Q: Why was `track` made mandatory in `@for`?
> `track` tells Angular how to identify items for efficient DOM reconciliation. Without it, Angular would destroy and recreate all DOM nodes on every change. Mandatory `track` prevents accidental performance bugs.

### Q: What is the difference between `input()` and `@Input()`?
| | `@Input()` | `input()` (Signal) |
|---|---|---|
| Type | Decorator | Function |
| Change tracking | Zone.js / `ngOnChanges` | Signal dependency graph |
| Required | `@Input({ required: true })` | `input.required<T>()` |
| Default value | Field initializer | `input<T>(defaultValue)` |
| Reactive | Only via `ngOnChanges` | `computed()`, `effect()` |

---

## Resources
- Official Docs: [angular.dev](https://angular.dev)
- Migration Guide: [angular.dev/update-guide](https://angular.dev/update-guide)
- Angular Blog: [blog.angular.dev](https://blog.angular.dev)
- Signals RFC: [github.com/angular/angular/discussions/49685](https://github.com/angular/angular/discussions/49685)